# Modello: ViT encoder + GPT-2 decoder

Costruiamo il modello di image captioning seguendo l'architettura encoder-decoder
classica per task multimodali:

- **Encoder**: ViT (Vision Transformer) pre-addestrato. Riceve un'immagine 224×224
  e produce 197 vettori di feature (1 [CLS] + 14×14 patch da 16 pixel).
- **Decoder**: GPT-2 small pre-addestrato. Riceve i token già generati e, tramite
  cross-attention, "guarda" le feature visive per produrre il token successivo.

Hugging Face fornisce la classe `VisionEncoderDecoderModel` che combina i due in
modo trasparente, gestendo automaticamente la cross-attention.

## Strategia
1. Caricare i due modelli pre-addestrati e combinarli
2. Creare una classe Dataset PyTorch per Flickr8k
3. Creare il DataLoader con batching efficiente
4. Test forward+backward su un singolo batch (sanity check)
5. Salvare la "configurazione" del modello pronta per il training (notebook 03)

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import (
    VisionEncoderDecoderModel,
    GPT2Tokenizer,
    ViTImageProcessor,
)

# Riproducibilita'
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Path del progetto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "flickr8k"
IMAGES_DIR = DATA_DIR / "Images"
SPLIT_FILE = DATA_DIR / "captions_with_split.csv"

# Iperparametri (decisi nel notebook 01)
MAX_LENGTH = 30
BATCH_SIZE = 32  # provvisorio, lo ricalibriamo dopo aver visto la VRAM usata

# Modelli pre-addestrati (riferimenti standard)
ENCODER_NAME = "google/vit-base-patch16-224-in21k"
DECODER_NAME = "gpt2"

print(f"\nEncoder: {ENCODER_NAME}")
print(f"Decoder: {DECODER_NAME}")
print(f"Max length: {MAX_LENGTH} token")

Device: cuda
GPU: NVIDIA GeForce RTX 5070
VRAM: 12.8 GB

Encoder: google/vit-base-patch16-224-in21k
Decoder: gpt2
Max length: 30 token


In [2]:
# VisionEncoderDecoderModel: combina automaticamente encoder ViT + decoder GPT-2
# aggiungendo i layer di cross-attention nel decoder
print("Caricamento del modello combinato (prima volta: download ~1.3 GB)...")
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    ENCODER_NAME,
    DECODER_NAME,
)

# Tokenizer e image processor
tokenizer = GPT2Tokenizer.from_pretrained(DECODER_NAME)
image_processor = ViTImageProcessor.from_pretrained(ENCODER_NAME)

# Configurazione del decoder GPT-2:
# - GPT-2 non ha pad_token nativo. Usiamo eos_token come pad.
tokenizer.pad_token = tokenizer.eos_token

# Diciamo al modello quali token usare per:
# - decoder_start_token_id: cosa generare per primo (BOS)
# - pad_token_id: token da ignorare nella loss
# - eos_token_id: token che segnala "fine sequenza"
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Iperparametri di generazione (li useremo solo in fase di inference, ma settiamoli ora)
model.config.max_length = MAX_LENGTH
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.num_beams = 4

# Stampa il numero di parametri
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParametri totali: {total_params/1e6:.1f}M")
print(f"Parametri trainabili: {trainable_params/1e6:.1f}M")

# Sposta su GPU
model = model.to(device)
print(f"Modello su: {next(model.parameters()).device}")

Caricamento del modello combinato (prima volta: download ~1.3 GB)...


Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at gpt2 and are newly initialized: ['h.0.crossattention.c_attn.bias', 'h.0.crossattention.c_attn.weight', 'h.0.crossattention.c_proj.bias', 'h.0.crossattention.c_proj.weight', 'h.0.crossattention.q_attn.bias', 'h.0.crossattention.q_attn.weight', 'h.0.ln_cross_attn.bias', 'h.0.ln_cross_attn.weight', 'h.1.crossattention.c_attn.bias', 'h.1.crossattention.c_attn.weight', 'h.1.crossattention.c_proj.bias', 'h.1.crossattention.c_proj.weight', 'h.1.crossattention.q_attn.bias', 'h.1.crossattention.q_attn.weight', 'h.1.ln_cross_attn.bias', 'h.1.ln_cross_attn.weight', 'h.10.crossattention.c_attn.bias', 'h.10.crossattention.c_attn.weight', 'h.10.crossattention.c_proj.bias', 'h.10.crossattention.c_proj.weight', 'h.10.crossattention.q_attn.bias', 'h.10.crossattention.q_attn.weight', 'h.10.ln_cross_attn.bias', 'h.10.ln_cross_attn.weight', 'h.11.crossattention.c_attn.bias', 'h.11.crossattention.c_attn.weight', 'h.11.crossat


Parametri totali: 239.2M
Parametri trainabili: 239.2M
Modello su: cuda:0


## Dataset class: Flickr8kCaptionsDataset

PyTorch richiede che ogni dataset implementi due metodi:
- `__len__`: quante coppie (immagine, didascalia) ci sono
- `__getitem__(idx)`: restituisce l'esempio numero `idx` già preparato

Per il nostro task, ogni esempio deve restituire:
- `pixel_values`: tensore (3, 224, 224) — l'immagine processata da ViT
- `labels`: tensore (max_length,) — i token della didascalia, padding incluso

Il modello durante il training calcola la loss confrontando le sue predizioni
con `labels`, ignorando le posizioni di padding.

In [3]:
class Flickr8kCaptionsDataset(Dataset):
    """Dataset PyTorch per Flickr8k.
    
    Ogni elemento e' una coppia (immagine, didascalia) gia' processata.
    """
    
    def __init__(self, df_split, images_dir, image_processor, tokenizer, max_length=30):
        """
        Args:
            df_split: DataFrame con colonne 'image' e 'caption' (gia' filtrato per split)
            images_dir: Path della cartella che contiene i .jpg
            image_processor: ViTImageProcessor di HuggingFace
            tokenizer: GPT2Tokenizer di HuggingFace
            max_length: lunghezza massima della sequenza tokenizzata
        """
        self.df = df_split.reset_index(drop=True)
        self.images_dir = images_dir
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Carica e processa l'immagine
        img_path = self.images_dir / row['image']
        image = Image.open(img_path).convert("RGB")
        pixel_values = self.image_processor(image, return_tensors="pt")["pixel_values"]
        pixel_values = pixel_values.squeeze(0)  # rimuovi la dim batch fittizia
        
        # 2. Tokenizza la didascalia
        # Aggiungiamo BOS all'inizio e EOS alla fine, paddiamo a max_length
        caption = row['caption']
        text = self.tokenizer.bos_token + " " + caption + " " + self.tokenizer.eos_token
        
        encoding = self.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"].squeeze(0)        # (max_length,)
        attention_mask = encoding["attention_mask"].squeeze(0)  # (max_length,) 1=token reale, 0=pad

        # Mascheriamo SOLO il padding (non tutti gli <|endoftext|>):
        # usiamo attention_mask invece di confrontare con pad_token_id
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        
        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }


# Carichiamo lo split che abbiamo salvato nel notebook 01
df = pd.read_csv(SPLIT_FILE)
print(f"Dataset totale: {len(df)} esempi")
print(f"Distribuzione split:\n{df['split'].value_counts()}\n")

# Crea i tre dataset
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

train_dataset = Flickr8kCaptionsDataset(train_df, IMAGES_DIR, image_processor, tokenizer, MAX_LENGTH)
val_dataset = Flickr8kCaptionsDataset(val_df, IMAGES_DIR, image_processor, tokenizer, MAX_LENGTH)
test_dataset = Flickr8kCaptionsDataset(test_df, IMAGES_DIR, image_processor, tokenizer, MAX_LENGTH)

print(f"Train: {len(train_dataset)} esempi")
print(f"Val:   {len(val_dataset)} esempi")
print(f"Test:  {len(test_dataset)} esempi")

Dataset totale: 40455 esempi
Distribuzione split:
split
train    30000
test      5455
val       5000
Name: count, dtype: int64

Train: 30000 esempi
Val:   5000 esempi
Test:  5455 esempi


In [4]:
# Estraiamo un singolo esempio e verifichiamo le shape
sample = train_dataset[0]
print("Forma di un singolo esempio:")
print(f"  pixel_values: {sample['pixel_values'].shape}, dtype={sample['pixel_values'].dtype}")
print(f"  labels:       {sample['labels'].shape}, dtype={sample['labels'].dtype}")
print()
print(f"  Range pixel_values: [{sample['pixel_values'].min():.3f}, {sample['pixel_values'].max():.3f}]")
print(f"  Primi 15 token labels: {sample['labels'][:15].tolist()}")
print(f"  Ultimi 15 token labels (-100 = pad ignorato): {sample['labels'][-15:].tolist()}")
print()

# Decodifichiamo i token (sostituendo -100 con pad per leggibilita')
labels_for_decode = sample['labels'].clone()
labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id
decoded = tokenizer.decode(labels_for_decode, skip_special_tokens=False)
print(f"  Didascalia decodificata: {decoded!r}")

Forma di un singolo esempio:
  pixel_values: torch.Size([3, 224, 224]), dtype=torch.float32
  labels:       torch.Size([30]), dtype=torch.int64

  Range pixel_values: [-0.851, 1.000]
  Primi 15 token labels: [50256, 317, 2042, 3290, 290, 257, 13489, 3290, 389, 4330, 220, 50256, -100, -100, -100]
  Ultimi 15 token labels (-100 = pad ignorato): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

  Didascalia decodificata: '<|endoftext|> A black dog and a spotted dog are fighting <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>'


## DataLoader

Il `DataLoader` raggruppa gli esempi in batch e gestisce:
- **shuffling**: ordine casuale a ogni epoca (solo train)
- **batching**: stack di tensori in (batch_size, ...)
- **multi-worker loading**: 4 worker leggono immagini in parallelo dal disco
  mentre la GPU lavora sul batch corrente, evitando il bottleneck I/O
- **pin_memory**: pre-allocazione in RAM "pinned" → trasferimento CPU→GPU piu' veloce

In [5]:
NUM_WORKERS = 0  # adatta in base al tuo PC; 4 e' sicuro per quasi tutti

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,           # mescola ad ogni epoca
    num_workers=NUM_WORKERS,
    pin_memory=True,
    #persistent_workers=True,  # tieni i worker vivi tra le epoche (no re-spawn ogni volta)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # per la validazione l'ordine non conta
    num_workers=NUM_WORKERS,
    pin_memory=True,
    #persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    #persistent_workers=True,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Batch size: 32
Train batches: 938
Val batches:   157
Test batches:  171


In [6]:
import time

# Estraiamo un batch
print("Caricamento del primo batch (i worker si spawnano ora, puo' richiedere ~10s)...")
t0 = time.time()
batch = next(iter(train_loader))
elapsed = time.time() - t0
print(f"Tempo: {elapsed:.1f}s")
print()
print(f"Forme del batch:")
print(f"  pixel_values: {batch['pixel_values'].shape}")
print(f"  labels:       {batch['labels'].shape}")
print()
print(f"Memoria stimata di un batch: {(batch['pixel_values'].element_size() * batch['pixel_values'].nelement() + batch['labels'].element_size() * batch['labels'].nelement()) / 1e6:.1f} MB")

Caricamento del primo batch (i worker si spawnano ora, puo' richiedere ~10s)...
Tempo: 0.2s

Forme del batch:
  pixel_values: torch.Size([32, 3, 224, 224])
  labels:       torch.Size([32, 30])

Memoria stimata di un batch: 19.3 MB


In [7]:
import time

# Spostiamo il batch sulla GPU
pixel_values = batch['pixel_values'].to(device)
labels = batch['labels'].to(device)

print(f"Input pixel_values shape: {pixel_values.shape}")
print(f"Input labels shape: {labels.shape}")
print()

# Misuriamo VRAM PRIMA del forward
torch.cuda.empty_cache()
mem_before = torch.cuda.memory_allocated() / 1e9
print(f"VRAM occupata prima del forward: {mem_before:.2f} GB")

# === Forward pass ===
print("\nForward pass...")
t0 = time.time()
outputs = model(pixel_values=pixel_values, labels=labels)
torch.cuda.synchronize()  # aspetta che la GPU finisca davvero
forward_time = time.time() - t0

loss = outputs.loss
logits = outputs.logits

print(f"  Tempo forward: {forward_time*1000:.0f} ms")
print(f"  Loss: {loss.item():.4f}")
print(f"  Logits shape: {logits.shape}  (batch, seq_len, vocab_size)")

mem_after_fwd = torch.cuda.memory_allocated() / 1e9
print(f"  VRAM dopo forward: {mem_after_fwd:.2f} GB")

# === Backward pass ===
print("\nBackward pass...")
t0 = time.time()
loss.backward()
torch.cuda.synchronize()
backward_time = time.time() - t0
print(f"  Tempo backward: {backward_time*1000:.0f} ms")

mem_after_bwd = torch.cuda.memory_allocated() / 1e9
print(f"  VRAM dopo backward: {mem_after_bwd:.2f} GB (modello + attivazioni + gradienti)")

# Pulisci i gradienti per non lasciare residui in memoria
model.zero_grad()
torch.cuda.empty_cache()

# === Sanity check ===
import math
expected_loss_random = math.log(50257)
print()
print("=== Sanity check ===")
print(f"  Loss e' un numero finito: {torch.isfinite(loss).item()}")
print(f"  Loss > 0: {loss.item() > 0}")
print(f"  Loss attesa per modello casuale: ~{expected_loss_random:.2f}  (cross-entropy uniforme su 50257 token)")
print(f"  La nostra loss: {loss.item():.4f}")

Input pixel_values shape: torch.Size([32, 3, 224, 224])
Input labels shape: torch.Size([32, 30])

VRAM occupata prima del forward: 1.00 GB

Forward pass...
  Tempo forward: 645 ms
  Loss: 6.2096
  Logits shape: torch.Size([32, 30, 50257])  (batch, seq_len, vocab_size)
  VRAM dopo forward: 6.82 GB

Backward pass...
  Tempo backward: 238 ms
  VRAM dopo backward: 2.34 GB (modello + attivazioni + gradienti)

=== Sanity check ===
  Loss e' un numero finito: True
  Loss > 0: True
  Loss attesa per modello casuale: ~10.82  (cross-entropy uniforme su 50257 token)
  La nostra loss: 6.2096


## Risultati del sanity check

✅ Forward pass funzionante: loss = 6.21 (finita, positiva, ragionevole)
✅ Backward pass funzionante: gradienti calcolati, VRAM liberata
✅ Forme tensoriali corrette: logits (B, T, V) come atteso

**Note interpretative:**
- La loss iniziale (6.21) e' inferiore al baseline random (10.82) perche'
  il decoder GPT-2 pre-addestrato fornisce gia' un buon prior linguistico
  anche senza usare correttamente le feature visive.
- Target di loss in fine training: ~2.0-2.5 (riferimento standard Flickr8k).

**Consumo VRAM (batch_size=32):**
- Picco di forward: ~6.8 GB / 12.8 GB disponibili → margine sano
- Manteniamo batch_size=32 come scelta standard in letteratura

## Salvataggio dello stato iniziale

Salviamo la "configurazione di partenza" del modello prima del training.
Cosi' il notebook 03 puo' ricaricare esattamente questo punto di partenza
senza dover riscaricare i pesi pre-addestrati ogni volta.

In [8]:
import json

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Salva il modello (architettura + pesi)
INITIAL_MODEL_DIR = CHECKPOINT_DIR / "model_initial"
print(f"Salvataggio modello in {INITIAL_MODEL_DIR}...")
model.save_pretrained(INITIAL_MODEL_DIR)

# Salva tokenizer e image_processor (utili per inference)
tokenizer.save_pretrained(INITIAL_MODEL_DIR)
image_processor.save_pretrained(INITIAL_MODEL_DIR)

# Salva un piccolo file di metadati con gli iperparametri
metadata = {
    "encoder_name": ENCODER_NAME,
    "decoder_name": DECODER_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "vocab_size": tokenizer.vocab_size,
    "total_params_M": round(total_params / 1e6, 2),
    "trainable_params_M": round(trainable_params / 1e6, 2),
    "initial_loss": round(loss.item(), 4),
}

with open(CHECKPOINT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata salvata in {CHECKPOINT_DIR / 'metadata.json'}:")
print(json.dumps(metadata, indent=2))

# Lista dei file salvati
print(f"\nFile salvati in {INITIAL_MODEL_DIR}:")
for f in sorted(INITIAL_MODEL_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:>8.1f} MB")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 30, 'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warning will be raised to an exception 

Salvataggio modello in c:\dev\image-captioning\checkpoints\model_initial...

Metadata salvata in c:\dev\image-captioning\checkpoints\metadata.json:
{
  "encoder_name": "google/vit-base-patch16-224-in21k",
  "decoder_name": "gpt2",
  "max_length": 30,
  "batch_size": 32,
  "num_workers": 0,
  "vocab_size": 50257,
  "total_params_M": 239.2,
  "trainable_params_M": 239.2,
  "initial_loss": 6.2096
}

File salvati in c:\dev\image-captioning\checkpoints\model_initial:
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  merges.txt                                    0.5 MB
  model.safetensors                           956.8 MB
  preprocessor_config.json                      0.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer_config.json                         0.0 MB
  vocab.json                                    1.0 MB
